# 02 - Limpieza
Aplicacion de las reglas documentadas en docs/methodology.md sobre el subconjunto
de suicidio 2023 (9,072 registros, ya filtrado en 01_profiling.ipynb).

Reglas a aplicar:
1. Recodificar codigos de 'no especificado'/'se ignora' a NaN explicito (no se imputa)
2. Separar la variable Edad (mezcla unidad + valor)
3. Traducir catalogos clave a etiquetas legibles (Sexo, causa CIE-10, entidad)
4. Verificar resultados y guardar en data/processed/

In [ ]:
import pandas as pd
import os
import sys
sys.path.append('../src')
from cleaning_utils import (
    load_dbf, normalize_columns, CATALOG_CANONICAL_COLUMNS, null_summary, dtype_summary,
    recode_null_codes, split_edad, translate_catalog
)


## 1. Cargar el subconjunto filtrado (handoff de 01_profiling.ipynb)

In [ ]:
df = pd.read_csv('../data/interim/suicidio_2023_filtrado.csv', encoding='utf-8', low_memory=False, dtype=str)
print(f'Registros cargados: {len(df):,}')
df.head()


## 2. Cargar catalogos necesarios para traducir etiquetas

In [ ]:
cat_geo = load_dbf('../data/raw/CATEMLDE23.dbf')
cat_geo = normalize_columns(cat_geo, canonical_names=CATALOG_CANONICAL_COLUMNS)
cat_causa = load_dbf('../data/raw/CATMINDE.dbf')
cat_causa = normalize_columns(cat_causa, canonical_names=CATALOG_CANONICAL_COLUMNS)
print('Catalogo geografico:', cat_geo.shape)
print('Catalogo causa CIE-10:', cat_causa.shape)
cat_geo.head()


## 3. Regla 1 - Recodificar 'no especificado'/'se ignora' a NaN explicito
Decision documentada en methodology.md: NO se imputa, se preserva el hueco como NaN real.

In [ ]:
print('Nulos ANTES de recodificar:')
print(null_summary(df).head(10))

df = recode_null_codes(df)

print('\nNulos DESPUES de recodificar:')
print(null_summary(df).head(10))


## 4. Regla 2 - Separar la variable Edad (unidad + valor)
Edad mezcla horas/dias/meses/anios en un solo codigo numerico. Se crean edad_valor y edad_unidad,
conservando la columna original Edad sin modificar.

In [ ]:
df = split_edad(df, col='Edad')
df[['Edad', 'edad_valor', 'edad_unidad']].head(10)


In [ ]:
df['edad_unidad'].value_counts(dropna=False)


## 5. Regla 3 - Traducir catalogos a etiquetas legibles
Se agregan columnas nuevas descriptivas; se conservan los codigos originales.

In [ ]:
# Sexo: etiqueta simple, no requiere catalogo externo
df['sexo_desc'] = df['Sexo'].map({'1': 'Hombre', '2': 'Mujer'})
df['sexo_desc'].value_counts(dropna=False)


In [ ]:
# Causa de defuncion (CIE-10 detallado) via catalogo CATMINDE
df = translate_catalog(df, df_col='Causa_def', catalog_df=cat_causa,
                        catalog_code_col='Cve', catalog_desc_col='Descrip',
                        new_col_name='causa_def_desc')
df[['Causa_def', 'causa_def_desc']].drop_duplicates().head(10)


In [ ]:
# Entidad de ocurrencia via catalogo geografico
# El catalogo geografico combina entidad+municipio+localidad en Cve_ent+Cve_mun+Cve_loc;
# para el nombre de la ENTIDAD, filtramos donde Cve_mun == '000' y Cve_loc == '0000'
cat_entidades = cat_geo[(cat_geo['Cve_mun'] == '000') & (cat_geo['Cve_loc'] == '0000')]
df = translate_catalog(df, df_col='Ent_ocurr', catalog_df=cat_entidades,
                        catalog_code_col='Cve_ent', catalog_desc_col='Nom_loc',
                        new_col_name='ent_ocurr_desc')
df[['Ent_ocurr', 'ent_ocurr_desc']].drop_duplicates().sort_values('Ent_ocurr').head(15)


## 6. Verificacion antes de guardar
Confirmar que el volumen de registros no cambio (la limpieza no debe filtrar filas, solo corregir valores).

In [ ]:
print(f'Registros: {len(df):,} (debe seguir siendo 9,072)')
print(f'Columnas: {df.shape[1]} (74 originales + nuevas: edad_valor, edad_unidad, sexo_desc, causa_def_desc, ent_ocurr_desc)')
df.head()


## 7. Guardar dataset limpio en data/processed/

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/suicidio_2023_limpio.csv', index=False, encoding='utf-8')
print(f'Guardado: {len(df):,} registros x {df.shape[1]} columnas en data/processed/suicidio_2023_limpio.csv')


## 8. Hallazgos y decisiones de la fase de limpieza
_Documentar aqui: cuantos valores se recodificaron por columna, distribucion de edad_unidad,
cualquier caso inesperado encontrado. Trasladar a docs/methodology.md y docs/quality_report.md._